# Employee Performance Scoring Model

This is a theoretical representation of the model; the parameters I will be using are:

1. Task & Deliverable Performance
2. Quality of Work
3. Reliability & Consistency
4. Collaboration & Communication
5. Initiative, Ownership & Problem-Solving
6. Learning, Growth & Adaptability
7. Time & Resource Efficiency
8. Goal Alignment & Strategic Contribution
9. Leadership & Influence
10. Cultural Fit & Organisational Citizenship
11. Availability, Attendance & Workload Health
12. Client & Stakeholder Impact

Within each parameter I introduce multiple variables to capture every potential scenario in an employee's performance. Each variable is assigned a specific weight to calculate a precise metric for that parameter. Finally, by weighting the parameters themselves, the model produces a robust, comprehensive score that accounts for most of the edge cases as well.


## Helper Functions

Before computing any parameter, we define a library of reusable mathematical tools that the scoring engine relies on throughout:

| Function | Purpose |
| --- | --- |
| `bci` (Bayesian Credible Interval) | Given observed successes and trials, returns a conservative lower-bound estimate from a Beta posterior, so that small sample sizes are automatically penalised. Used wherever a stated rate is grounded in evidence (on-time completion, first-pass acceptance, follow-through, attendance, etc.). |
| `bcifromrate` | Convenience wrapper that converts a rate plus an assumed sample size into a `bci` call. |
| `edp` (Exponential Decay Penalty) | Computes $$\exp(-k \cdot x)$$ — a smooth, tuneable decay. Used for penalties such as average days late, defect rates, response latency, and overtime ratios. |
| `lnnorm` (Logarithmic Normalisation) | Computes $$\log(1+\text{count}) / \log(1+\text{cap})$$ — maps an integer count into [0, 1] with diminishing returns. Used for initiative counts, cross-team contributions, skills acquired, mentoring counts, voluntary activities, and positive client feedback. |
| `gaussianpenalty` | Computes $$\exp(-(x-\text{ideal})^2 / 2\sigma^2)$$ — penalises deviation from an ideal value with a bell-shaped curve, for variables where both "too high" and "too low" are undesirable (workload ratio, leave ratio). |
| `hrc` (Huber Robust Composite) | A robust weighted average: it computes an initial weighted mean, then iteratively down-weights any component whose value deviates from the composite by more than δ. This prevents a single outlier variable from disproportionately dragging a parameter score up or down. It is the primary aggregation function for every parameter. |


In [22]:
from scipy.stats import beta as beta_dist
import numpy as np

def bci(successes, trials, confidence=0.05):
    if trials == 0:
        return 0.0
    s = successes if isinstance(successes, (int, float)) else successes
    n = trials
    return beta_dist.ppf(confidence, s + 1, n - s + 1)

def bci_from_rate(rate, assumed_n=50):
    s = round(rate * assumed_n)
    return bci(s, assumed_n)

def edp(x, k):
    return np.exp(-k * x)

def ln_norm(count, cap):
    if cap <= 0:
        return 0.0
    return min(np.log(1 + count) / np.log(1 + cap), 1.0)

def gaussian_penalty(actual, ideal, sigma):
    return np.exp(-((actual - ideal) ** 2) / (2 * sigma ** 2))

def hrc(values, weights, delta=0.15, max_iter=100, tol=1e-6):
    values = np.array(values, dtype=float)
    weights = np.array(weights, dtype=float)
    weights = weights / weights.sum() 
    theta = np.dot(weights, values)
    for _ in range(max_iter):
        residuals = np.abs(values - theta)
        adjusted_weights = weights.copy()
        for i in range(len(values)):
            if residuals[i] > delta:
                adjusted_weights[i] = weights[i] * (delta / residuals[i])
        if adjusted_weights.sum() == 0:
            break
        theta_new = np.dot(adjusted_weights, values) / adjusted_weights.sum()
        
        if abs(theta_new - theta) < tol:
            theta = theta_new
            break
        theta = theta_new
    return theta

## Model Setup — Robust Aggregators, Input Data & Global Controls

This cell completes the helper library with `gaussianpenalty` and `hrc`, then defines the full input dataset and the two global control structures.

- **`gaussianpenalty`** rewards proximity to an ideal value and penalises deviation in either direction with a bell-shaped curve.
- **`hrc`** is the Huber Robust Composite described above — the workhorse aggregator for all twelve parameters.
- **`data`** is a list of twelve dictionaries, one per parameter, each holding that parameter's raw inputs. Every variable is annotated inline with its meaning, an example, and its valid range.
- **`gates`** is a list of seven binary knockout flags (`'f'` = pass, `'t'` = triggered). Any single trigger zeroes the final score.
- **`modifiers`** is a list of five continuous situational multipliers [M1 … M5] that scale the final score for data quality and evaluation-context realities.


In [23]:
data = [

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 1 — TASK & DELIVERABLE PERFORMANCE
    # "Does this employee get their work done, on time, and right?"
    # ═══════════════════════════════════════════════════════════════
    {
        # How many tasks did the employee complete on time, out of all tasks assigned?
        # Example: completed 44 out of 50 tasks on time → 0.88
        # [0.0 to 1.0]
        "tc": 0.88,

        # How many tasks did the employee actually complete in this evaluation period? (just the count)
        # [whole number, 0 or more]
        "tvnum": 52,

        # How many tasks is a person in this role EXPECTED to complete in the same period? (the benchmark/target)
        # [whole number, 1 or more]
        "tvbench": 45,

        # How many deliverables were submitted on time, out of all deliverables?
        # Example: 17 out of 20 deliverables on time → 0.85
        # [0.0 to 1.0]
        "dd": 0.85,

        # Total number of deliverables submitted.
        # [whole number, 1 or more]
        "tot_del": 20,

        # For the deliverables that WERE late, how many days late were they on average?
        # If nothing was late, enter 0.
        # Example: 3 late deliverables, averaging 2.5 days late → 2.5
        # [0 or more, can be decimal like 1.5]
        "adraw": 2.5,

        # Out of all HIGH-PRIORITY tasks, how many were completed on time?
        # Example: 9 out of 10 high-priority tasks done on time → 0.90
        # [0.0 to 1.0]
        "pr": 0.90,

        # How many tasks had to be ESCALATED to someone else (manager, senior, etc.) because the employee couldn't handle them, out of total tasks?
        # Example: 4 escalated out of 50 total → 0.08
        # [0.0 to 1.0] — lower is better
        "escraw": 0.08,

        # How many deliverables had to be sent back for REWORK (redo / major corrections), out of total completed?
        # Example: 5 reworked out of 50 completed → 0.10
        # [0.0 to 1.0] — lower is better
        "rwraw": 0.10,
    },

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 2 — QUALITY OF WORK
    # "How good is the actual output this employee produces?"
    # ═══════════════════════════════════════════════════════════════
    {
        # Overall quality rating given by manager / tech lead / peer reviewer, combined into one score.
        # 0 = terrible quality, 1 = exceptional quality
        # If you have a rating out of 5, divide by 5.
        # Example: 4.1 out of 5 → 0.82
        # [0.0 to 1.0]
        "qa": 0.82,

        # How many submissions were accepted on the FIRST TRY (no revisions needed), out of total submissions?
        # Example: 39 accepted first time out of 50 → 0.78
        # [0.0 to 1.0]
        "fa": 0.78,

        #Total number of submissions made.
        # [whole number, 1 or more]
        "tot_sub": 50,

        # On average, how many errors / defects / bugs were found per deliverable?
        # Example: 60 defects across 50 deliverables → 1.2
        # [0 or more, can be decimal]
        "drraw": 1.2,

        # How well does the employee follow standards, processes, SOPs, coding guidelines, or regulatory requirements?
        # 0 = ignores all standards, 1 = perfect compliance
        # [0.0 to 1.0]
        "cs": 0.85,

        # How thorough and detailed is the work?
        # Does the employee cover edge cases, think things through?
        # 0 = superficial / sloppy, 1 = exceptionally thorough
        # [0.0 to 1.0]
        "dt": 0.80,

        # How creative or innovative is the employee in solving problems? 
        # Does the employee come up with new approaches?
        # 0 = purely follows instructions, 1 = highly innovative
        # If creativity is NOT relevant to this role, enter 0.50
        # [0.0 to 1.0]
        "cr": 0.65,
    },

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 3 — RELIABILITY & CONSISTENCY
    # "Can you count on this person to deliver predictably,
    #  month after month, without babysitting?"
    # ═══════════════════════════════════════════════════════════════
    {
        # How much does this employee's performance FLUCTUATE from month to month?
        # This is the standard deviation of their monthly scores.
        # 0.00 = perfectly consistent every month
        # 0.05 = very consistent
        # 0.15 = moderate ups and downs
        # 0.30 = wildly inconsistent
        # If you don't know, estimate based on gut feel.
        # [0.0 to 0.50 typically]
        "cvsigma": 0.12,

        # How much does the manager TRUST this employee to deliver without checking in?
        # 0 = no trust at all, 1 = complete trust
        # [0.0 to 1.0]
        "dp": 0.85,

        # When this employee says "I'll do it", how often do they actually follow through and complete it?
        # Example: followed through on 18 out of 20 commitments → 0.90
        # [0.0 to 1.0]
        "fs": 0.90,

        # Total number of commitments made by the employee.
        # # [whole number, 1 or more]
        "tot_com": 20,

        # During critical crunch periods (deadlines, launches, emergencies), how often was this employee available and present?
        # Example: available for 4 out of 5 critical periods → 0.80
        # [0.0 to 1.0]
        "av": 0.80,

        # Total number of critical periods faced from when the employee joined.
        # [whole number, 1 or more]
        "tot_cri": 5,

        # Pick a representative task. How many days did the employee ESTIMATE it would take?
        # [any positive number, can be decimal]
        "pmest": 5.0,

        # For that same task, how many days did it ACTUALLY take?
        # [any positive number, can be decimal]
        "pmactual": 6.0,
    },

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 4 — COLLABORATION & COMMUNICATION
    # "How well does this person work with others and share info?"
    # ═══════════════════════════════════════════════════════════════
    {
        # How much does this employee contribute to TEAM goals (not just their own tasks)?
        # Rated by peers.
        # 0 = works in isolation, 1 = exceptional team player
        # [0.0 to 1.0]
        "tc4": 0.80,

        # How clearly does this employee communicate? (emails, meetings, documentation, presentations)
        # 0 = confusing and unclear, 1 = crystal clear communicator
        # [0.0 to 1.0]
        "cm": 0.85,

        # On average, how many HOURS does it take for this employee to respond to messages, emails, or requests from colleagues?
        # Example: usually replies within 3 hours → 3.0
        # [0 or more, can be decimal like 1.5]
        "rsphours": 3.0,

        # How well does this employee handle disagreements and conflicts at work?
        # 0 = creates or escalates conflict, 1 = resolves issues constructively
        # [0.0 to 1.0]
        "cf": 0.75,

        # How well does this employee share knowledge with others?
        # (writing docs, mentoring juniors, giving presentations, helping teammates learn)
        # 0 = hoards knowledge, 1 = actively shares and teaches
        # [0.0 to 1.0]
        "kn": 0.70,

        # How well does this employee RECEIVE feedback?
        # Do they listen, accept, and actually improve based on it?
        # 0 = defensive / ignores feedback, 1 = embraces and acts on it
        # [0.0 to 1.0]
        "fb": 0.80,

        # Total number of handoffs occured.
        # [whole number, 1 or more]
        "tot_han": 60,
        
        # How often does this employee BLOCK others from doing their work?
        # (holding up reviews, not responding to handoffs, being a bottleneck)
        # Example: blocked others 3 times out of 60 handoff
        # opportunities → 3/60 = 0.05
        # [0.0 to 1.0] — lower is better
        "blraw": 0.05,
    },

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 5 — INITIATIVE, OWNERSHIP & PROBLEM-SOLVING
    # "Does this employee go beyond what's assigned and take
    #  ownership of outcomes?"
    # ═══════════════════════════════════════════════════════════════
    {
        # How many improvements, ideas, or initiatives did the employee START ON THEIR OWN (not assigned by anyone) during this evaluation period?
        # Example: suggested and implemented 4 improvements → 4
        # [whole number, 0 or more]
        "incount": 4,

        # How much ownership does this employee take?
        # Do they see things through from start to finish, or hand off problems to others?
        # 0 = avoids responsibility, 1 = full end-to-end ownership
        # [0.0 to 1.0]
        "ow": 0.85,

        # How good is this employee at solving problems?
        # Do they find ROOT CAUSES, or just apply band-aids?
        # 0 = can't solve problems independently,
        # 1 = excellent analytical problem-solver
        # [0.0 to 1.0]
        "ps": 0.80,

        # How independently can this employee work?
        # How much supervision do they need?
        # 0 = needs constant hand-holding,
        # 1 = fully self-directed
        # [0.0 to 1.0]
        "au": 0.82,

        # How many times did the employee contribute meaningfully to ANOTHER TEAM's work? (cross-functional contributions)
        # Example: helped 2 other teams → 2
        # [whole number, 0 or more]
        "bccount": 2,

        # Did the employee's self-initiated improvements actually produce measurable results?
        # 0 = no real impact, 1 = significant measurable outcomes
        # [0.0 to 1.0]
        "imp": 0.70,
    },

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 6 — LEARNING, GROWTH & ADAPTABILITY
    # "Is this employee growing and adapting, or standing still?"
    # ═══════════════════════════════════════════════════════════════
    {
        # How many NEW SKILLS or CERTIFICATIONS did the employee acquire during this period?
        # (courses completed, certs earned, new tools mastered)
        # [whole number, 0 or more]
        "skcount": 2,

        # How actively does the employee pursue professional development? (attending training, reading, conferences, self-study)
        # 0 = zero interest in development,
        # 1 = actively and consistently developing
        # [0.0 to 1.0]
        "pd": 0.70,

        # How well does the employee adapt to CHANGE?
        # (new tools, new processes, shifting requirements, team restructuring)
        # 0 = resists all change, 1 = thrives on change
        # [0.0 to 1.0]
        "ad6": 0.80,

        # In the LAST evaluation, were areas for improvement identified? If so, how much did the employee actually improve in those areas?
        # 0 = no improvement at all,
        # 1 = fully addressed all previous feedback
        # If this is the first evaluation, enter 0.50
        # [0.0 to 1.0]
        "fp": 0.75,

        # How quickly does this employee become productive in
        # NEW areas compared to peers?
        # 0 = very slow learner, 1 = picks things up immediately
        # [0.0 to 1.0]
        "lr": 0.70,

        # How many DIFFERENT ROLES or FUNCTIONS can this employee competently perform?
        # Example: can do their own role + 2 adjacent ones → 3
        # [whole number, 1 or more]
        "vrroles": 2,
    },

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 7 — TIME & RESOURCE EFFICIENCY
    # "Is this employee productive with their time and resources?"
    # ═══════════════════════════════════════════════════════════════
    {
        # What fraction of available work hours are actually spent on productive work?
        # Example: 6.8 productive hours out of 8 available → 0.85
        # [0.0 to 1.0]
        "ut": 0.85,

        # How much output does this employee produce per unit of effort, compared to role benchmarks?
        # 0 = far below benchmark, 1 = at or above benchmark
        # [0.0 to 1.0]
        "et": 0.80,

        # How efficiently does the employee use company resources?
        # (budget, tools, software licenses, assets)
        # 0 = wasteful, 1 = very efficient
        # [0.0 to 1.0]
        "rt": 0.75,

        # Total number of meetings happened.
        # [whole number, 1 or more]
        "tot_meet": 20,

        # Out of all meetings attended, how many were actually productive and valuable?
        # Example: 14 productive meetings out of 20 total → 0.70
        # [0.0 to 1.0]
        "mt": 0.70,

        # What fraction of this employee's hours are OVERTIME, compared to standard hours?
        # Example: 4 overtime hours per 40 standard hours → 0.10
        # NOTE: Some overtime is normal. Excessive overtime signals inefficiency or unsustainable workload.
        # [0.0 or more, typically 0.0 to 0.50]
        "otratio": 0.15,

        # What fraction of total work hours were WASTED or IDLE?
        # (non-productive time, waiting, unnecessary rework, unproductive browsing)
        # Example: 1 hour wasted out of 8 → 0.125
        # [0.0 to 1.0] — lower is better
        "wmraw": 0.12,
    },

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 8 — GOAL ALIGNMENT & STRATEGIC CONTRIBUTION
    # "Is this employee's work advancing what the company
    #  actually cares about?"
    # ═══════════════════════════════════════════════════════════════
    {
        # What fraction of the employee's tasks are directly linked to company goals / OKRs / KPIs?
        # Example: 42 out of 50 tasks mapped to OKRs → 0.84
        # [0.0 to 1.0]
        "ga": 0.85,

        # Total number of tasks assigned.
        # [whole number, 1 or more]
        "tot_task": 50,

        # Total number of individual goals assigned.
        # [whole number, 1 or more]
        "tot_goal": 5,

        # What fraction of the employee's INDIVIDUAL GOALS were met or exceeded?
        # Example: hit 4 out of 5 goals → 0.80
        # [0.0 to 1.0]
        "gc": 0.80,

        # How well does the employee UNDERSTAND the company's strategy and connect their daily work to it?
        # 0 = no awareness, 1 = deeply strategic thinker
        # [0.0 to 1.0]
        "st": 0.75,

        # Does the employee consistently work on the MOST IMPORTANT things first?
        # 0 = poor prioritisation, 1 = always works on highest impact
        # [0.0 to 1.0]
        "pv": 0.78,

        # Has the employee's work had a VISIBLE, MEASURABLE impact on team or business metrics?
        # 0 = no visible impact, 1 = clearly moved the needle
        # [0.0 to 1.0]
        "vi": 0.72,
    },

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 9 — LEADERSHIP & INFLUENCE
    # "Does this person make the people around them better?
    #  (with or without a manager title)"
    # ═══════════════════════════════════════════════════════════════
    {
        # How many people is the employee actively MENTORING or coaching (formally or informally)?
        # [whole number, 0 or more]
        "mncount": 2,

        # How much INFLUENCE does this employee have on others?
        # Do people adopt their ideas, follow their standards, look to them for guidance?
        # 0 = no influence, 1 = widely respected thought leader
        # [0.0 to 1.0]
        "inf": 0.70,

        # How effectively does this employee DELEGATE work to others?
        # If the employee has NO direct reports and doesn't delegate, enter 0.50 (neutral — not applicable)
        # 0 = terrible at delegating, 1 = excellent delegation
        # [0.0 to 1.0]
        "de": 0.50,

        # How good are the DECISIONS this employee makes within their authority?
        # 0 = poor judgment, 1 = consistently excellent decisions
        # [0.0 to 1.0]
        "dm": 0.80,

        # Do the people who work WITH or AROUND this employee perform BETTER because of them?
        # 0 = drags the team down, 1 = significantly elevates team
        # [0.0 to 1.0]
        "te": 0.75,

        # How does this employee perform UNDER PRESSURE?
        # (during crises, incidents, tight deadlines, emergencies)
        # 0 = falls apart, 1 = thrives and leads through crisis
        # [0.0 to 1.0]
        "cr9": 0.72,
    },

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 10 — CULTURAL FIT & ORGANISATIONAL CITIZENSHIP
    # "Does this employee align with company values and
    #  contribute to a positive workplace?"
    # ═══════════════════════════════════════════════════════════════
    {
        # How well does the employee's behaviour align with the company's stated values?
        # 0 = actively contradicts values, 1 = embodies them fully
        # [0.0 to 1.0]
        "va": 0.85,

        # How ethical and honest is this employee?
        # (integrity, transparency, trustworthiness)
        # 0 = dishonest / unethical, 1 = impeccable integrity
        # [0.0 to 1.0]
        "eth": 0.90,

        # Does this employee contribute to a POSITIVE work environment? (supportive, inclusive, constructive)
        # 0 = toxic presence, 1 = everyone's better off with them
        # [0.0 to 1.0]
        "po": 0.80,

        # How many VOLUNTARY, non-mandatory activities did the employee participate in?
        # (social events, committees, charity drives, team outings, culture initiatives)
        # [whole number, 0 or more]
        "evcount": 3,

        # How many CONFIRMED negative behavioural incidents were recorded? 
        # (HR complaints, policy violations, disciplinary actions)
        #  WARNING: 5 or more incidents = this entire parameter
        #             score drops to ZERO.
        # [whole number, 0 or more] — lower is better
        "ngcount": 0,

        # How professional is this employee's general conduct?
        # (punctuality to meetings, email etiquette, dress code, respectful communication)
        # 0 = unprofessional, 1 = exemplary professionalism
        # [0.0 to 1.0]
        "prc": 0.88,
    },

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 11 — AVAILABILITY, ATTENDANCE & WORKLOAD HEALTH
    # "Is the employee present, and is their work pattern
    #  sustainable?"
    # ═══════════════════════════════════════════════════════════════
    {
        # Out of total working days (AFTER removing approved leave), how many days was the employee actually present?
        # Example: present 190 out of 200 working days → 0.95
        # [0.0 to 1.0]
        "at": 0.95,

        # Out of total working days, how many days did the employee arrive ON TIME?
        # Example: on time 180 out of 200 days → 0.90
        # [0.0 to 1.0]
        "pu": 0.90,

        # What fraction of working days were UNPLANNED absences?
        # (calling in sick last minute, no-shows, unexplained absences)
        # Example: 10 unplanned absence days out of 200 → 0.05
        # [0.0 to 1.0] — lower is better
        "uaraw": 0.05,

        # How much BURNOUT RISK does this employee show?
        # Consider: overtime trends, unused leave, engagement scores, self-reported stress, withdrawal from team activities.
        # 0.00 = no burnout signs at all
        # 0.20 = mild signs
        # 0.50 = moderate burnout risk
        # 0.80 = severe burnout risk
        # 1.00 = critical burnout
        # [0.0 to 1.0] — lower is better
        "bn": 0.20,

        # What is the employee's ACTUAL workload compared to the IDEAL workload?
        # 1.00 = perfectly balanced (doing exactly the right amount)
        # 0.70 = underloaded (only doing 70% of ideal capacity)
        # 1.30 = overloaded (doing 130% of ideal capacity)
        # [0.5 to 2.0 typically, 1.0 is ideal]
        "wlratio": 1.15,

        # Total number of woking days for that employee.
        # [whole number, 1 or more]
        "tot_days": 200,
        
        # What fraction of their available leave did the employee actually TAKE?
        # Example: took 16 leave days out of 200 working days → 0.08
        # [0.0 to 1.0]
        "lvratio": 0.07,

        # What is the company's TARGET leave ratio?
        # (how much leave does the company WANT employees to take?)
        # Example: company expects ~8% of working days as leave → 0.08
        # [0.0 to 1.0]
        "lvtarget": 0.08,
    },

    # ═══════════════════════════════════════════════════════════════
    # PARAMETER 12 — CLIENT & STAKEHOLDER IMPACT
    # "What value did this employee deliver to clients or
    #  internal stakeholders?"
    # ═══════════════════════════════════════════════════════════════
    {
        # How satisfied are clients / stakeholders with this employee's work specifically?
        # 0 = very dissatisfied, 1 = extremely satisfied
        # If you have a rating out of 5, divide by 5.
        # Example: average 4.1 out of 5 → 0.82
        # [0.0 to 1.0]
        "csat": 0.82,

        # How many times did a client or stakeholder give POSITIVE feedback specifically mentioning this employee?
        # (thank-you emails, commendations, shout-outs)
        # [whole number, 0 or more]
        "cfbcount": 5,

        # Total number of client accounts, this employee worked on.
        # [whole number, 1 or more]
        "tot_acc": 20,
        
        # Out of all client accounts this employee worked on, how many were RETAINED or EXPANDED?
        # Example: 17 retained out of 20 total → 0.85
        # If not applicable (no client accounts), enter 0.50
        # [0.0 to 1.0]
        "ret": 0.85,

        # How much revenue is DIRECTLY ATTRIBUTABLE to this employee's work? (in your currency)
        # If not applicable or not tracked, enter the same value as the target below (so the ratio = 1.0, neutral)
        # [any number, 0 or more]
        "revattributed": 1100000,

        # What was the revenue TARGET for this employee?
        # [any number, 1 or more]
        "revtarget": 1000000,

        # How many NEGATIVE client interactions were recorded?
        # (complaints, escalations from clients, formal grievances)
        # WARNING: 7 or more incidents significantly tanks this score.
        # [whole number, 0 or more] — lower is better
        "nccount": 1,

        # How good is this employee at building lasting professional RELATIONSHIPS with clients / stakeholders beyond just the current project?
        # 0 = purely transactional, 1 = builds deep, lasting trust
        # [0.0 to 1.0]
        "re": 0.75,
    },
]

gates=['f','f','f','f','f','f','f']
modifiers = [1.0,1.0,1.0,0.9,1.0]

## Parameter 1 — Task & Deliverable Performance

> *"Does this employee get their work done, on time, and right?"*

This parameter measures raw delivery: how reliably tasks and deliverables are completed on time, how the throughput compares to role expectations, and how much friction the work generates through lateness, escalations, and rework.

**How it is computed**

- **Timeliness (Tc)** uses a BCI grounded in the implied number of tasks (`tvnum / tc`), so a high on-time rate on a thin task count stays conservative.
- **Throughput (Tv)** compares actual completions to the role benchmark, capped at 1.3× and normalised, so over-delivery is rewarded but bounded.
- **Deliverable timeliness (Dd)** and **high-priority timeliness (Pr)** are BCI scores over the deliverable and high-priority counts.
- **Lateness penalty (Ad)** decays exponentially with average days late.
- **Escalation (Esc)** and **rework (Rw)** are drag terms: their incident counts are reconstructed from the raw rates, scored via BCI, and inverted (1 − BCI) so lower is better.
- A positive core (**Tcore**) and a drag sub-score (**Tdrag**) are each aggregated via `hrc`, then blended with throughput.

**Composite Formula**

```
Tcore = HRC([Tc, Dd, Ad, Pr], [0.30, 0.25, 0.20, 0.25])
Tdrag = HRC([Esc, Rw], [0.50, 0.50])
X1    = 0.60·Tcore + 0.20·Tv + 0.20·Tdrag
```


In [4]:
p1 = data[0] 

Tc = bci(p1["tvnum"], round( p1["tvnum"]/ p1["tc"]))

Tv = min(p1["tvnum"] / p1["tvbench"], 1.3) / 1.3
assumed_deliverables = p1["tot_del"]
Dd = bci(round(p1["dd"] * assumed_deliverables), assumed_deliverables)
Ad = edp(p1["adraw"], 0.15)
assumed_hp = 10
Pr = bci(round(p1["pr"] * assumed_hp), assumed_hp)
esc_count = round(p1["escraw"] * p1["tvnum"])
Esc = 1.0 - bci(esc_count, p1["tvnum"])
rw_count = round(p1["rwraw"] * p1["tvnum"])
Rw = 1.0 - bci(rw_count, p1["tvnum"])
T_core = hrc(
    values=[Tc, Dd, Ad, Pr],
    weights=[0.30, 0.25, 0.20, 0.25]
)

T_drag = hrc(
    values=[Esc, Rw],
    weights=[0.50, 0.50]
)

X1 = 0.60 * T_core + 0.20 * Tv + 0.20 * T_drag

print(X1)

0.7899407076108276


## Parameter 2 — Quality of Work

> *"How good is the actual output this employee produces?"*

This parameter captures the intrinsic quality of the output: reviewer ratings, first-pass acceptance, defect density, adherence to standards, thoroughness, and creativity.

**How it is computed**

- **First-pass acceptance (Fa)** is a BCI over total submissions, penalising high acceptance claims on small samples.
- **Defect rate (Dr)** decays exponentially with the average defects per deliverable.
- **Quality rating (Qa)**, **standards compliance (Cs)**, and **thoroughness (Dt)** enter directly as stated [0,1] scores.
- The five quality-output components are aggregated via `hrc` into **Qout**.
- **Creativity (Cr)** is held separate and blended in at 15%, since it is role-dependent and shouldn't dominate the core quality signal.

**Composite Formula**

```
Q_out = HRC([Qa, Fa, Dr, Cs, Dt], [0.25, 0.20, 0.20, 0.15, 0.20])
X2   = 0.85·Qout + 0.15·Cr
```


In [24]:
p2 = data[1]

Qa = p2["qa"]  

assumed_submissions = p2["tot_sub"]
Fa = bci(round(p2["fa"] * assumed_submissions), assumed_submissions)
Dr = edp(p2["drraw"], 0.4)
Cs = p2["cs"]   
Dt = p2["dt"] 
Cr = p2["cr"] 

Q_out = hrc(
    values=[Qa, Fa, Dr, Cs, Dt],
    weights=[0.25, 0.20, 0.20, 0.15, 0.20]
)

X2 = 0.85 * Q_out + 0.15 * Cr

print(X2)

0.7349355257598978


## Parameter 3 — Reliability & Consistency

> *"Can you count on this person to deliver predictably, month after month, without babysitting?"*

This parameter measures dependability: how stable performance is over time, how much the manager trusts the employee, follow-through on commitments, availability during crunch periods, and estimation accuracy.

**How it is computed**

- **Consistency (Cv)** transforms the month-to-month standard deviation via $$\exp(-2\cdot\sigma)$$, so low variance scores near 1.0 and high variance is penalised smoothly.
- **Follow-through (Fs)** and **critical availability (Av)** are BCI scores over their respective commitment and critical-period counts.
- **Dependability (Dp)** enters directly as a stated trust score.
- **Estimation accuracy (Pm)** is $$1 - \min(|\text{actual} - \text{estimate}| / \text{estimate}, 1)$$, rewarding tight estimates and capping the penalty at a full miss.
- All five components are aggregated via `hrc`.

**Composite Formula**

```
Cv = exp(-2·cvsigma)
Pm = 1 - min(|pmactual - pmest| / pmest, 1)
X3 = HRC([Cv, Dp, Fs, Av, Pm], [0.25, 0.25, 0.25, 0.10, 0.15])
```


In [25]:
p3 = data[2]

Cv = np.exp(-2 * np.sqrt(p3["cvsigma"] ** 2))
Dp = p3["dp"]  

assumed_commitments = p3["tot_com"]
Fs = bci(round(p3["fs"] * assumed_commitments), assumed_commitments)


assumed_critical = p3["tot_cri"]
Av = bci(round(p3["av"] * assumed_critical), assumed_critical)

Pm = 1.0 - min(abs(p3["pmactual"] - p3["pmest"]) / p3["pmest"], 1.0)

X3 = hrc(
    values=[Cv, Dp, Fs, Av, Pm],
    weights=[0.25, 0.25, 0.25, 0.10, 0.15]
)

print(X3)

0.7739100410124575


## Parameter 4 — Collaboration & Communication

> *"How well does this person work with others and share info?"*

This parameter measures teamwork: contribution to team goals, clarity of communication, responsiveness, conflict handling, knowledge sharing, feedback reception, and whether the employee blocks others.

**How it is computed**

- **Responsiveness (Rsp)** decays exponentially with average response hours.
- **Team contribution (Tc4)**, **communication (Cm)**, **conflict handling (Cf)**, **knowledge sharing (Kn)**, and **feedback reception (Fb)** enter directly as stated scores.
- These six positive signals are aggregated via `hrc` into **Cpos**.
- **Blocking (Bl)** is a drag term: the block-incident count is reconstructed from the raw rate over total handoffs, scored via BCI, and inverted so lower is better.
- The positive composite is then suppressed multiplicatively by the blocking factor.

**Composite Formula**

```
Cpos = HRC([Tc4, Cm, Rsp, Cf, Kn, Fb], [0.25, 0.20, 0.15, 0.10, 0.15, 0.15])
Bl   = 1 - BCI(blockcount, totalhandoffs)
X4   = Cpos · Bl
```


In [26]:
p4 = data[3]

Tc4 = p4["tc4"]  

Cm = p4["cm"]     

Rsp = edp(p4["rsphours"], 0.05)

Cf = p4["cf"]   

Kn = p4["kn"]    
Fb = p4["fb"]      

assumed_handoffs = p4["tot_han"]
bl_count = round(p4["blraw"] * assumed_handoffs)
Bl = 1.0 - bci(bl_count, assumed_handoffs)

C_pos = hrc(
    values=[Tc4, Cm, Rsp, Cf, Kn, Fb],
    weights=[0.25, 0.20, 0.15, 0.10, 0.15, 0.15]
)

X4 = C_pos * Bl

print(X4)

0.7809611074902966


## Parameter 5 — Initiative, Ownership & Problem-Solving

> *"Does this employee go beyond what's assigned and take ownership of outcomes?"*

This parameter measures proactivity: how many improvements the employee starts, how much end-to-end ownership they take, problem-solving depth, independence, cross-team contribution, and the measurable impact of their initiatives.

**How it is computed**

- **Initiative count (In)** is normalised via `lnnorm(incount, 10)` and **cross-team contributions (Bc)** via `lnnorm(bccount, 5)`, both with diminishing returns.
- **Ownership (Ow)**, **problem-solving (Ps)**, **autonomy (Au)**, and **impact (Imp)** enter directly as stated scores.
- All six components are aggregated via `hrc`.

**Composite Formula**

```
In = lnnorm(incount, 10)
Bc = lnnorm(bccount, 5)
X5 = HRC([In, Ow, Ps, Au, Bc, Imp], [0.20, 0.25, 0.20, 0.15, 0.10, 0.10])
```


In [27]:
p5 = data[4]

In = ln_norm(p5["incount"], 10)
Ow = p5["ow"]     

Ps = p5["ps"]    

Au = p5["au"]    

Bc = ln_norm(p5["bccount"], 5)

Imp = p5["imp"]  

X5 = hrc(
    values=[In, Ow, Ps, Au, Bc, Imp],
    weights=[0.20, 0.25, 0.20, 0.15, 0.10, 0.10]
)

print(X5)

0.7610522675707938


## Parameter 6 — Learning, Growth & Adaptability

> *"Is this employee growing and adapting, or standing still?"*

This parameter measures development: new skills acquired, active pursuit of professional growth, adaptability to change, follow-through on prior feedback, learning speed, and versatility across roles.

**How it is computed**

- **Skills acquired (Sk)** is normalised via `lnnorm(skcount, 5)` with diminishing returns.
- **Versatility (Vr)** is the number of roles the employee can perform, capped and normalised against a target of 3.
- **Professional development (Pd)**, **adaptability (Ad6)**, **feedback follow-through (Fp)**, and **learning rate (Lr)** enter directly as stated scores.
- All six components are aggregated via `hrc`.

**Composite Formula**

```
Sk = lnnorm(skcount, 5)
Vr = min(vrroles / 3, 1)
X6 = HRC([Sk, Pd, Ad6, Fp, Lr, Vr], [0.20, 0.15, 0.25, 0.20, 0.10, 0.10])
```


In [9]:
p6 = data[5]

Sk = ln_norm(p6["skcount"], 5)

Pd = p6["pd"]      
Ad6 = p6["ad6"]  
Fp = p6["fp"]   
Lr = p6["lr"]   

Vr = min(p6["vrroles"] / 3.0, 1.0)

X6 = hrc(
    values=[Sk, Pd, Ad6, Fp, Lr, Vr],
    weights=[0.20, 0.15, 0.25, 0.20, 0.10, 0.10]
)

print(X6)


0.7142961052197584


## Parameter 7 — Time & Resource Efficiency

> *"Is this employee productive with their time and resources?"*

This parameter measures efficiency: utilisation of available hours, output per unit of effort, resource stewardship, meeting productivity, overtime burden, and wasted time.

**How it is computed**

- **Meeting productivity (Mt)** is a BCI over the total meeting count.
- **Overtime (Ot)** decays exponentially on the excess above a 10% allowance (`max(0, otratio − 0.10)`), so modest overtime is tolerated and excess is penalised.
- **Wasted time (Wm)** is the simple complement `1 − wmraw`, so lower waste scores higher.
- **Utilisation (Ut)**, **efficiency (Et)**, and **resource use (Rt)** enter directly as stated scores.
- All six components are aggregated via `hrc`.

**Composite Formula**

```
Ot = exp(-2·max(0, otratio - 0.10))
Wm = 1 - wmraw
X7 = HRC([Ut, Et, Rt, Mt, Ot, Wm], [0.25, 0.25, 0.15, 0.10, 0.15, 0.10])
```


In [28]:
p7 = data[6]

Ut = p7["ut"]     

Et = p7["et"]  

Rt = p7["rt"] 

assumed_meetings = p7["tot_meet"]
Mt = bci(round(p7["mt"] * assumed_meetings), assumed_meetings)
Ot = edp(max(0, p7["otratio"] - 0.10), 2.0)

Wm = 1.0 - p7["wmraw"]


X7 = hrc(
    values=[Ut, Et, Rt, Mt, Ot, Wm],
    weights=[0.25, 0.25, 0.15, 0.10, 0.15, 0.10]
)

print(X7)

0.8152506745557503


## Parameter 8 — Goal Alignment & Strategic Contribution

> *"Is this employee's work advancing what the company actually cares about?"*

This parameter measures strategic value: how tightly tasks map to company goals, individual goal attainment, strategic understanding, prioritisation, and visible impact on outcomes.

**How it is computed**

- **Goal alignment (Ga)** is a BCI over total tasks, and **goal completion (Gc)** is a BCI over total individual goals, both grounding stated rates in their counts.
- **Strategic understanding (St)**, **prioritisation (Pv)**, and **visible impact (Vi)** enter directly as stated scores.
- All five components are aggregated via `hrc`.

**Composite Formula**

```
Ga = BCI(round(ga·tottask), tottask)
Gc = BCI(round(gc·totgoal), totgoal)
X8 = HRC([Ga, Gc, St, Pv, Vi], [0.25, 0.30, 0.15, 0.15, 0.15])
```


In [29]:
p8 = data[7]

assumed_tasks_8 = p8["tot_task"]
Ga = bci(round(p8["ga"] * assumed_tasks_8), assumed_tasks_8)

assumed_goals = p8["tot_goal"]
Gc = bci(round(p8["gc"] * assumed_goals), assumed_goals)

St = p8["st"]   

Pv = p8["pv"]

Vi = p8["vi"] 
X8 = hrc(
    values=[Ga, Gc, St, Pv, Vi],
    weights=[0.25, 0.30, 0.15, 0.15, 0.15]
)

print(X8)

0.6802587131672205


## Parameter 9 — Leadership & Influence

> *"Does this person make the people around them better — with or without a manager title?"*

This parameter measures leadership equity: mentoring reach, influence over others, delegation skill, decision quality, team elevation, and performance under pressure.

**How it is computed**

- **Mentoring (Mn)** is normalised via `lnnorm(mncount, 4)` with diminishing returns.
- **Influence (Inf)**, **delegation (De)**, **decision-making (Dm)**, **team elevation (Te)**, and **crisis performance (Cr9)** enter directly as stated scores.
- All six components are aggregated via `hrc`.

**Composite Formula**

```
Mn = lnnorm(mncount, 4)
X9 = HRC([Mn, Inf, De, Dm, Te, Cr9], [0.15, 0.20, 0.15, 0.20, 0.20, 0.10])
```


In [30]:
p9 = data[8]

Mn = ln_norm(p9["mncount"], 4)

Inf = p9["inf"] 

De = p9["de"]   

Dm = p9["dm"]     

Te = p9["te"]  

Cr9 = p9["cr9"] 

X9 = hrc(
    values=[Mn, Inf, De, Dm, Te, Cr9],
    weights=[0.15, 0.20, 0.15, 0.20, 0.20, 0.10]
)

print(X9)

0.7081069567145918


## Parameter 10 — Cultural Fit & Organisational Citizenship

> *"Does this employee align with company values and contribute to a positive workplace?"*

This parameter measures cultural contribution: values alignment, ethics and integrity, positive presence, voluntary participation, professional conduct, and a hard penalty for confirmed negative incidents.

**How it is computed**

- **Voluntary activities (Ev)** are normalised via `lnnorm(evcount, 6)` with diminishing returns.
- **Values alignment (Va)**, **ethics (Eth)**, **positivity (Po)**, and **professional conduct (Prc)** enter directly as stated scores.
- These five positive signals are aggregated via `hrc` into **Ccit**.
- **Negative incidents (Ng)** apply a steep linear penalty `max(0, 1 − 0.2·ngcount)` — five or more incidents zero this factor entirely — applied multiplicatively to the citizenship composite.

**Composite Formula**

```
Ev   = lnnorm(evcount, 6)
Ng   = max(0, 1 - 0.2·ngcount)
Ccit = HRC([Va, Eth, Po, Ev, Prc], [0.25, 0.25, 0.20, 0.15, 0.15])
X10  = Ng · Ccit
```


In [31]:
p10 = data[9]

Va = p10["va"]

Eth = p10["eth"]

Po = p10["po"]

Ev = ln_norm(p10["evcount"], 6)

Ng = max(0, 1.0 - 0.2 * p10["ngcount"])

Prc = p10["prc"] 

C_cit = hrc(
    values=[Va, Eth, Po, Ev, Prc],
    weights=[0.25, 0.25, 0.20, 0.15, 0.15]
)

X10 = Ng * C_cit

print(X10)

0.8363621561324067


## Parameter 11 — Availability, Attendance & Workload Health

> *"Is the employee present, and is their work pattern sustainable?"*

This parameter measures presence and sustainability: attendance, punctuality, unplanned absences, burnout risk, workload balance, and whether leave usage matches the company target.

**How it is computed**

- **Attendance (At)** and **punctuality (Pu)** are BCI scores over total working days.
- **Unplanned absence (Ua)** is a drag term: the absence count is reconstructed from the raw rate, scored via BCI, and inverted so lower is better.
- **Burnout (Bn)** is the complement `1 − bn`, so fewer burnout signs score higher.
- **Workload balance (Wl)** uses a Gaussian penalty centred on the ideal ratio of 1.0, penalising both under- and over-loading.
- **Leave usage (Lv)** uses a Gaussian penalty centred on the company's target leave ratio, rewarding healthy leave-taking near target.
- All six components are aggregated via `hrc`.

**Composite Formula**

```
Ua  = 1 - BCI(absencecount, totdays)
Bn  = 1 - bn
Wl  = gaussianpenalty(wlratio, 1.0, 0.20)
Lv  = gaussianpenalty(lvratio, lvtarget, 0.15)
X11 = HRC([At, Pu, Ua, Bn, Wl, Lv], [0.25, 0.15, 0.20, 0.15, 0.15, 0.10])
```


In [34]:
p11 = data[10]

assumed_workdays = p11["tot_days"]
At = bci(round(p11["at"] * assumed_workdays), assumed_workdays)

Pu = bci(round(p11["pu"] * assumed_workdays), assumed_workdays)

ua_count = round(p11["uaraw"] * assumed_workdays)
Ua = 1.0 - bci(ua_count, assumed_workdays)

Bn = 1.0 - p11["bn"]

Wl = gaussian_penalty(p11["wlratio"], 1.0, 0.20)

Lv = gaussian_penalty(p11["lvratio"], p11["lvtarget"], 0.15)

X11 = hrc(
    values=[At, Pu, Ua, Bn, Wl, Lv],
    weights=[0.25, 0.15, 0.20, 0.15, 0.15, 0.10]
)

print(X11)

0.8848790119626456


## Parameter 12 — Client & Stakeholder Impact

> *"What value did this employee deliver to clients or internal stakeholders?"*

This parameter measures external value: client satisfaction, positive feedback volume, account retention, revenue attribution, relationship-building, and a penalty for negative client interactions.

**How it is computed**

- **Positive feedback (Cfb)** is normalised via `lnnorm(cfbcount, 10)` with diminishing returns.
- **Retention (Ret)** is a BCI over total client accounts.
- **Revenue (Rev)** is attributed revenue over target, capped at 1.2× and normalised, so exceeding target is rewarded but bounded.
- **Satisfaction (Csat)** and **relationship-building (Re)** enter directly as stated scores.
- These five components are aggregated via `hrc` into **Simpact**.
- **Negative interactions (Nc)** apply a linear penalty `max(0, 1 − 0.15·nccount)` — seven or more incidents tank this score — applied multiplicatively.

**Composite Formula**

```
Cfb     = lnnorm(cfbcount, 10)
Rev     = min(revattributed / revtarget, 1.2) / 1.2
Nc      = max(0, 1 - 0.15·nccount)
Simpact = HRC([Csat, Cfb, Ret, Rev, Re], [0.25, 0.15, 0.20, 0.20, 0.20])
X12     = Nc · Simpact
```


In [35]:
p12 = data[11]

Csat = p12["csat"] 

Cfb = ln_norm(p12["cfbcount"], 10)

assumed_accounts = p12["tot_acc"]
Ret = bci(round(p12["ret"] * assumed_accounts), assumed_accounts)

Rev = min(p12["revattributed"] / p12["revtarget"], 1.2) / 1.2

Nc = max(0, 1.0 - 0.15 * p12["nccount"])

Re = p12["re"]   

S_impact = hrc(
    values=[Csat, Cfb, Ret, Rev, Re],
    weights=[0.25, 0.15, 0.20, 0.20, 0.20]
)

X12 = Nc * S_impact

print(X12)

0.6668882314041245


## Hard Knockout Gates

Seven binary gates that can instantly zero the entire employee score. If any single gate fails, the score is invalidated regardless of how strong the other parameters are.

| Gate | Condition that fails the gate (sets it to 0) |
| --- | --- |
| G1 | Employee is on active suspension, a PIP with frozen evaluation, or a formal disciplinary hold. |
| G2 | Employee has submitted resignation or been given termination notice. |
| G3 | Role has been eliminated or is being made redundant. |
| G4 | Employee is on extended leave (medical, parental, sabbatical) with insufficient data for a meaningful evaluation. |
| G5 | Conflict of interest — employee is under investigation for fraud, embezzlement, or a material policy violation that makes scoring premature. |
| G6 | Data integrity failure — evaluation data has been flagged as fabricated, manipulated, or fundamentally unreliable. |
| G7 | Legal hold — employee is party to active litigation against the organisation where scoring could create legal exposure. |

Each gate is encoded as `'f'` (pass) or `'t'` (triggered / fail) in the `gates` list. The composite gate is:

$$G = G_1 \times G_2 \times G_3 \times G_4 \times G_5 \times G_6 \times G_7$$

Any single 0 makes $$G = 0$$, killing the final score.


In [36]:
g=[]
for i in range(len(gates)):
    h=gates[i]
    if h == 't':
        g.append(0)
    else:
        g.append(1)
G=1
for h in g:
    G=G*h
print(G)

1


## Hard Knockout Gates

Seven binary gates that can instantly zero the entire employee score. If any single gate fails, the score is invalidated regardless of how strong the other parameters are.

| Gate | Condition that fails the gate (sets it to 0) |
| --- | --- |
| G1 | Employee is on active suspension, a PIP with frozen evaluation, or a formal disciplinary hold. |
| G2 | Employee has submitted resignation or been given termination notice. |
| G3 | Role has been eliminated or is being made redundant. |
| G4 | Employee is on extended leave (medical, parental, sabbatical) with insufficient data for a meaningful evaluation. |
| G5 | Conflict of interest — employee is under investigation for fraud, embezzlement, or a material policy violation that makes scoring premature. |
| G6 | Data integrity failure — evaluation data has been flagged as fabricated, manipulated, or fundamentally unreliable. |
| G7 | Legal hold — employee is party to active litigation against the organisation where scoring could create legal exposure. |

Each gate is encoded as `'f'` (pass) or `'t'` (triggered / fail) in the `gates` list. The composite gate is:

$$G = G_1 \times G_2 \times G_3 \times G_4 \times G_5 \times G_6 \times G_7$$

Any single 0 makes $$G = 0$$, killing the final score.


In [37]:
a=[]
for i in range(len(modifiers)):
    h=modifiers[i]
    a.append(h)
M=1
for i in range(5):
    M=M*a[i]
print(M)

0.9


# Aggregation Functions — Choquet Integral

The final score combines all fifteen parameter scores using a Choquet integral over a non-additive (fuzzy) measure, which captures interaction effects between parameters that a simple weighted sum cannot express.

## Why a Choquet integral?

A weighted sum assumes each parameter contributes independently. In reality, parameters interact:

- **Synergies** — strong financial qualification combined with high quality and impact is worth more than the parts in isolation; certain capabilities reinforce each other.
- **Redundancies** — parameters that measure overlapping qualities (e.g. cultural fit and citizenship-style behaviours) should not be double-counted.

The Choquet integral handles this with a **capacity** (fuzzy measure) over all subsets of parameters. For tractability we use a **2-additive capacity**:

- **Singleton capacities `a_i`** — the base importance of each parameter.
- **Pairwise interaction coefficients `a_ij`** — positive values create synergy, negative values create redundancy.

This cell defines the supporting functions:

- **`sigmoid_transform`** — a logistic curve centred at `x0` with steepness `k`, used to spread mid-range scores for sharper discrimination before aggregation.
- **`mu`** — evaluates the capacity of any coalition (subset) of parameters by summing their singletons plus any active pairwise interactions.
- **`choquet_integral`** — sorts the scores ascending, then accumulates each marginal increment weighted by the capacity of the coalition of parameters scoring at least that high.


In [38]:
def sigmoid_transform(x, x0=0.50, k=8.0):
    return 1.0 / (1.0 + np.exp(-k * (x - x0)))
def mu(S, a, a_ij):
    value = sum(a[i] for i in S)
    for (i, j), coeff in a_ij.items():
        if i in S and j in S:
            value += coeff
    return value
def choquet_integral(scores, a, a_ij):
    n = len(scores)
    sigma = sorted(range(n), key=lambda i: scores[i])
    integral = 0.0
    for rank in range(n):
        coalition = set(sigma[rank:])
        mu_val = mu(coalition, a, a_ij)
        if rank == 0:
            delta = scores[sigma[rank]]
        else:
            delta = scores[sigma[rank]] - scores[sigma[rank - 1]]
        integral += delta * mu_val
    return integral

## Final Score — Choquet Integral Aggregation

The final score combines the twelve parameter scores using a **Choquet integral** over a non-additive (fuzzy) measure, which captures interaction effects between parameters that a simple weighted sum cannot express.

### Singleton capacities

| Parameter | $$a_i$$ | Rationale |
| --- | --- | --- |
| X1 — Task & Deliverable Performance | 0.120 | The strongest single signal — core output and delivery. |
| X2 — Quality of Work | 0.105 | Output is only valuable if it is good. |
| X3 — Reliability & Consistency | 0.090 | Predictable delivery without supervision. |
| X4 — Collaboration & Communication | 0.085 | Working effectively with others. |
| X5 — Initiative & Problem-Solving | 0.080 | Going beyond the assigned scope. |
| X6 — Learning & Adaptability | 0.070 | Forward-looking growth potential. |
| X7 — Time & Resource Efficiency | 0.065 | Productivity per unit of effort. |
| X8 — Goal Alignment & Strategic Contribution | 0.080 | Advancing what the company cares about. |
| X9 — Leadership & Influence | 0.055 | Elevating those around them. |
| X10 — Cultural Fit & Citizenship | 0.045 | Values alignment and positive presence. |
| X11 — Availability & Workload Health | 0.050 | Sustainable presence. |
| X12 — Client & Stakeholder Impact | 0.050 | External value delivered. |

> The singleton capacities sum to **0.895**, not 1.0. This is intentional — the model divides by $$\mu(N)$$ (the capacity of the full set) to normalise, so the unnormalised total does not need to equal 1.

### Pairwise interactions

| Pair | Coefficient | Interpretation |
| --- | --- | --- |
| (X1, X2) Task + Quality | +0.025 | Volume done and done well compounds (synergy). |
| (X1, X3) Task + Reliability | +0.020 | Consistent, predictable delivery (synergy). |
| (X5, X3) Initiative + Reliability | +0.015 | Self-starting and dependable (synergy). |
| (X2, X7) Quality + Efficiency | +0.015 | High-quality output produced efficiently (synergy). |
| (X5, X6) Initiative + Learning | +0.015 | Proactive growth mindset (synergy). |
| (X8, X12) Strategic + Client Impact | +0.020 | Strategic work that lands with stakeholders (synergy). |
| (X9, X10) Leadership + Citizenship | +0.010 | Influence rooted in strong values (synergy). |
| (X1, X7) Task + Efficiency | −0.010 | Both measure throughput — partial overlap (redundancy). |
| (X4, X9) Collaboration + Leadership | −0.010 | Influence and teamwork correlate (redundancy). |
| (X3, X11) Reliability + Availability | −0.010 | Presence and dependability partly overlap (redundancy). |
| (X6, X5) Learning + Initiative | −0.005 | Growth and proactivity correlate (slight redundancy). |

### Pre-processing before the integral

- Six parameter scores (**X4, X5, X6, X8, X9, X10** — indices `[3, 4, 5, 7, 8, 9]`) receive an additional sigmoid transform ($$x_0 = 0.50$$, $$k = 8.0$$) before entering the integral. These tend to cluster in a narrow mid-range, and the sigmoid spreads them for better discrimination.
- All scores are clamped to [0, 1] before aggregation.

### Final formula

```
Cμ   = ChoquetIntegral(Xfinal, a, aij)
μ(N) = mu(full set)                # normalisation constant
S    = 100 × G × M × (Cμ / μ(N))
```

The result is a score between 0 and 100, where:

- **G** (gates) can zero it instantly,
- **M** (modifiers) scales it for evaluation-context friction, and
- $$C_\mu / \mu(N)$$ is the Choquet-normalised composite of all twelve parameters.


In [39]:
X_raw = [X1, X2, X3, X4, X5, X6, X7, X8, X9, X10, X11, X12]
sigmoid_index=[3,4,5,7,8,9]
X_sig = [sigmoid_transform(x) if i in sigmoid_index else x for i, x in enumerate(X_raw)]
X_final = [max(0.0, min(1.0, x)) for x in X_sig]
a_vals = {0:0.120, 1:0.105, 2:0.090, 3:0.085, 4:0.080,
          5:0.070, 6:0.065, 7:0.080, 8:0.055, 9:0.045,
          10:0.050, 11:0.050}
a_ij_vals = {
    (0,1):+0.025, (0,2):+0.020, (4,2):+0.015, (1,6):+0.015,
    (4,5):+0.015, (7,11):+0.020, (8,9):+0.010,
    (0,6):-0.010, (3,8):-0.010, (2,10):-0.010, (5,4):-0.005,
}
C_mu = choquet_integral(X_final, a_vals, a_ij_vals)
mu_N = mu(set(range(12)), a_vals, a_ij_vals)
          
S = 100.0 * G * M * (C_mu / mu_N)

print(S)

73.03971163133609
